# ⚖️ Hybrid Ensemble — Accuracy (v1) + Safety (v4)
`Pred_Final = w × Model_A(absolute) + (1−w) × Model_B(physics+residual)`  
**Goal:** Find optimal `w` → Clarke A+B > 95% with Zone D < 3%.

---

In [ ]:
!pip install polars xgboost plotly pyarrow scikit-learn -q
import polars as pl, numpy as np, xgboost as xgb
import plotly.express as px, plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error
from datetime import datetime, timedelta
import os, gc, time, json

from google.colab import drive
drive.mount('/content/drive')
print(f"XGBoost: {xgb.__version__}")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
BASE  = "/content/drive/MyDrive/DiabetesKDS/data"
INPUT = f"{BASE}/processed/train_elite_physio_engineered.parquet"
MODEL_DIR = f"{BASE}/models"
os.makedirs(MODEL_DIR, exist_ok=True)

FEATS = ["CGM","IOB","COB","Vel_5min","Vel_15min","Acc_15min",
         "Roll_Mean_60","Roll_Std_60","Hour_Sin","Hour_Cos",
         "Insulin_Sensitivity_Risk"]
print(f"📁 {INPUT} — exists: {os.path.exists(INPUT)}")

---\n## STEP 1 — Load, Split, Weights

In [ ]:
df = pl.read_parquet(INPUT)
FEATS = [c for c in FEATS if c in df.columns]

# Physics baseline
df = df.with_columns(
    (pl.col("CGM") + pl.col("Vel_15min") * 2).alias("Linear_Forecast"),
)
df = df.with_columns(
    (pl.col("Target_CGM") - pl.col("Linear_Forecast")).alias("Target_Residual"),
)
df = df.sort("datetime").drop_nulls(
    subset=FEATS + ["Target_CGM", "Target_Residual", "Linear_Forecast"]
)
print(f"Data: {len(df):,} rows")

# Split
s = int(len(df) * 0.80)
tr, te = df.head(s), df.tail(len(df) - s)

X_tr = tr.select(FEATS).to_numpy().astype(np.float32)
X_te = te.select(FEATS).to_numpy().astype(np.float32)

y_tr_abs  = tr["Target_CGM"].to_numpy().astype(np.float32)
y_tr_res  = tr["Target_Residual"].to_numpy().astype(np.float32)
y_te_abs  = te["Target_CGM"].to_numpy().astype(np.float32)
lf_te     = te["Linear_Forecast"].to_numpy().astype(np.float32)

# Feature column indices for reconstruction
cgm_idx = FEATS.index("CGM")
vel15_idx = FEATS.index("Vel_15min")

test_ids = te["id"].to_list()
test_dt  = te["datetime"].to_list()

# Clinical weights (on absolute target)
def cw(y):
    w = np.ones_like(y, dtype=np.float32)
    w[y<70]=10.0; w[y>250]=5.0; w[(y>=180)&(y<=250)]=2.0
    return w

w_tr = cw(y_tr_abs); w_te = cw(y_te_abs)
del df, tr; gc.collect()
print(f"Train: {X_tr.shape} | Test: {X_te.shape}")

---\n## STEP 2 — Train Model A (Stabilizer: Absolute Target)

In [ ]:
params_shared = dict(
    objective="reg:squarederror", tree_method="hist", device="cuda",
    n_estimators=1500, early_stopping_rounds=50,
    eval_metric="rmse", random_state=42, verbosity=0,
    learning_rate=0.02, max_depth=10,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.0, reg_lambda=5.0, min_child_weight=10,
)

print("🚀 Training Model A (Absolute — Stabilizer)...")
t0 = time.time()
model_a = xgb.XGBRegressor(**params_shared)
model_a.fit(X_tr, y_tr_abs, sample_weight=w_tr,
    eval_set=[(X_te, y_te_abs)], sample_weight_eval_set=[w_te], verbose=False)
ta = time.time()-t0

pred_a = np.clip(model_a.predict(X_te), 20, 500)
rmse_a = np.sqrt(mean_squared_error(y_te_abs, pred_a))
print(f"✅ Model A done in {ta:.1f}s | Best iter: {model_a.best_iteration} | RMSE: {rmse_a:.4f}")

---\n## STEP 3 — Train Model B (Navigator: Physics Residual)

In [ ]:
print("🚀 Training Model B (Physics Residual — Navigator)...")
t0 = time.time()
model_b = xgb.XGBRegressor(**params_shared)
y_te_res = y_te_abs - lf_te  # test residual for eval_set
model_b.fit(X_tr, y_tr_res, sample_weight=w_tr,
    eval_set=[(X_te, y_te_res)], sample_weight_eval_set=[w_te], verbose=False)
tb = time.time()-t0

pred_b_res = model_b.predict(X_te)
pred_b = np.clip(lf_te + pred_b_res, 20, 500)  # Reconstruct
rmse_b = np.sqrt(mean_squared_error(y_te_abs, pred_b))
print(f"✅ Model B done in {tb:.1f}s | Best iter: {model_b.best_iteration} | RMSE: {rmse_b:.4f}")

print(f"\n📊 Individual Performance:")
print(f"   Model A (Absolute):  RMSE = {rmse_a:.4f}")
print(f"   Model B (Residual):  RMSE = {rmse_b:.4f}")

---\n## STEP 4 — Ensemble Optimization (The Blender) ⚖️

In [ ]:
def clarke_zones(ref, pred):
    ref, pred = np.asarray(ref,float), np.asarray(pred,float)
    z = np.full(len(ref), 'D', dtype='U1')
    for i in range(len(ref)):
        r,p = ref[i],pred[i]
        if (r<=70 and p<=70) or abs(p-r)<=0.2*r: z[i]='A'
        elif (r<=70 and p>=180) or (r>=180 and p<=70): z[i]='E'
        elif 70<=r<=180 and (p<70 or p>180): z[i]='C'
        elif (r<70 and 70<p<=180) or (r>180 and 70<=p<180): z[i]='D'
        else: z[i]='B'
    return z

def zone_stats(ref, pred):
    z = clarke_zones(ref, pred)
    uq, ct = np.unique(z, return_counts=True)
    d = dict(zip(uq, ct)); t = len(z)
    ab = (d.get('A',0)+d.get('B',0))/t*100
    zd = d.get('D',0)/t*100
    return ab, zd, d, t

print("✅ Clarke functions ready.")

In [ ]:
print("⚖️  Scanning ensemble weights...")
print(f"{'w_A':>5s} {'w_B':>5s} | {'RMSE':>8s} {'MAE':>8s} | {'A+B%':>7s} {'ZoneD%':>7s} | Status")
print("─" * 70)

results = []
for w in np.arange(0.0, 1.025, 0.05):
    w = round(w, 2)
    p_ens = np.clip(w * pred_a + (1-w) * pred_b, 20, 500)
    rmse_e = np.sqrt(mean_squared_error(y_te_abs, p_ens))
    mae_e  = mean_absolute_error(y_te_abs, p_ens)
    ab, zd, _, _ = zone_stats(y_te_abs, p_ens)
    
    flag = "✅" if ab > 95 and zd < 3 else ("🟡" if ab > 95 else "")
    print(f"{w:>5.2f} {1-w:>5.2f} | {rmse_e:>8.4f} {mae_e:>8.4f} | {ab:>6.2f}% {zd:>6.2f}% | {flag}")
    results.append({"w_a": w, "w_b": 1-w, "rmse": rmse_e, "mae": mae_e, "ab": ab, "zd": zd})

In [ ]:
# Find best weight: maximize A+B with Zone D < 3%
valid = [r for r in results if r["zd"] < 3.0]
if not valid:
    valid = results  # fallback: just maximize A+B
    print("⚠️  No weight achieves Zone D < 3%, selecting best A+B overall.")

best = max(valid, key=lambda x: x["ab"])
print(f"\n╔══════════════════════════════════════════════════╗")
print(f"║         🏆  OPTIMAL ENSEMBLE WEIGHT              ║")
print(f"╠══════════════════════════════════════════════════╣")
print(f"║  Model A (Absolute) weight : {best['w_a']:>5.2f}              ║")
print(f"║  Model B (Residual) weight : {best['w_b']:>5.2f}              ║")
print(f"║                                                  ║")
print(f"║  Clarke A+B : {best['ab']:>6.2f}%                          ║")
print(f"║  Zone D     : {best['zd']:>6.2f}%                          ║")
print(f"║  RMSE       : {best['rmse']:>8.4f} mg/dL                  ║")
print(f"║  MAE        : {best['mae']:>8.4f} mg/dL                  ║")
print(f"╚══════════════════════════════════════════════════╝")

In [ ]:
# Plot weight sweep
ws = [r["w_a"] for r in results]
fig = go.Figure()
fig.add_trace(go.Scatter(x=ws, y=[r["ab"] for r in results],
    name="Clarke A+B %", line=dict(color="#2ECC71", width=2.5)))
fig.add_trace(go.Scatter(x=ws, y=[r["zd"] for r in results],
    name="Zone D %", line=dict(color="#E74C3C", width=2.5)))
fig.add_trace(go.Scatter(x=ws, y=[r["rmse"] for r in results],
    name="RMSE", line=dict(color="#3498DB", width=2), yaxis="y2"))
fig.add_vline(x=best["w_a"], line_dash="dash", line_color="gold",
    annotation_text=f"Best w={best['w_a']:.2f}")
fig.add_hline(y=95, line_dash="dot", line_color="rgba(46,204,113,0.4)")
fig.add_hline(y=3, line_dash="dot", line_color="rgba(231,76,60,0.4)")

fig.update_layout(
    title="⚖️ Ensemble Weight Optimization",
    xaxis_title="Weight for Model A (Absolute)",
    yaxis_title="Percentage (%)",
    yaxis2=dict(title="RMSE (mg/dL)", overlaying="y", side="right"),
    template="plotly_dark", height=500,
    legend=dict(x=0.02, y=0.98),
)
fig.show()

---\n## STEP 5 — Final Clarke Error Grid (Best Ensemble)

In [ ]:
# Generate final ensemble prediction
pred_final = np.clip(best["w_a"] * pred_a + best["w_b"] * pred_b, 20, 500)
ab_f, zd_f, zdict, total = zone_stats(y_te_abs, pred_final)

# Full metrics
rmse_f = np.sqrt(mean_squared_error(y_te_abs, pred_final))
mae_f  = mean_absolute_error(y_te_abs, pred_final)
m = y_te_abs > 0
mape_f = np.mean(np.abs((y_te_abs[m]-pred_final[m])/y_te_abs[m]))*100
errs = np.abs(y_te_abs - pred_final)
w10,w15,w20 = [np.mean(errs<=t)*100 for t in [10,15,20]]

print("╔══════════════════════════════════════════════════════╗")
print("║       🩸 FINAL ENSEMBLE — CLARKE ERROR GRID        ║")
print("╠══════════════════════════════════════════════════════╣")
for zone in ['A','B','C','D','E']:
    n=zdict.get(zone,0); pct=n/total*100
    bar='█'*int(pct/2)
    print(f"║  Zone {zone}: {n:>10,} ({pct:>6.2f}%) {bar:20s}  ║")
print(f"║                                                      ║")
s = "✅ PASS" if ab_f > 95 else "⚠️  BELOW"
print(f"║  A+B = {ab_f:.2f}%  (target >95%)  {s:12s}      ║")
print(f"║  Zone D = {zd_f:.2f}%  (target <3%)                    ║")
print(f"║  v1 baseline = 92.91%  →  Δ = {ab_f-92.91:+.2f}%             ║")
print("╚══════════════════════════════════════════════════════╝")

In [ ]:
# Clarke Grid Plot
sn = min(100000, total)
ix = np.random.RandomState(42).choice(total, sn, replace=False)
zs = clarke_zones(y_te_abs[ix], pred_final[ix])
cl = {'A':'#2ECC71','B':'#F39C12','C':'#E74C3C','D':'#9B59B6','E':'#C0392B'}

fig = go.Figure()
fig.add_trace(go.Scattergl(x=y_te_abs[ix], y=pred_final[ix], mode='markers',
    marker=dict(size=2, color=[cl.get(z,'gray') for z in zs], opacity=0.4), hoverinfo='skip'))
fig.add_trace(go.Scatter(x=[0,400],y=[0,400], mode='lines',
    line=dict(color='white',width=1,dash='dash'), name='Perfect'))
xl = np.linspace(1,400,300)
for f in [1.2, 0.8]:
    fig.add_trace(go.Scatter(x=xl, y=xl*f, mode='lines',
        line=dict(color='rgba(46,204,113,0.4)'), showlegend=False))
for v in [70,180]:
    fig.add_hline(y=v, line_dash='dot', line_color='rgba(255,255,255,0.2)')
    fig.add_vline(x=v, line_dash='dot', line_color='rgba(255,255,255,0.2)')
fig.add_annotation(x=340,y=50, text=f"A+B: {ab_f:.1f}%",
    font=dict(size=16,color='white'), showarrow=False)
fig.update_layout(title=f"🩸 Clarke Grid — Ensemble (w_A={best['w_a']:.2f}, w_B={best['w_b']:.2f})",
    xaxis_title="Reference", yaxis_title="Predicted",
    template="plotly_dark", height=600, width=600,
    xaxis=dict(range=[0,400]), yaxis=dict(range=[0,400]))
fig.show()

In [ ]:
# ── 24h Prediction Trace (3 lines: A, B, Ensemble) ──────────────────
ta = np.array(test_ids)
uid,uc = np.unique(ta, return_counts=True)
bp = uid[np.argmax(uc)]; pm = ta==bp
pdt = np.array(test_dt)[pm]
mid = len(pdt)//2
wm = (pdt>=pdt[mid]) & (pdt<=pdt[mid]+timedelta(hours=24))

fig = go.Figure()
fig.add_trace(go.Scatter(x=pdt[wm], y=y_te_abs[pm][wm], name='Actual',
    line=dict(color='#00D4AA', width=2.5)))
fig.add_trace(go.Scatter(x=pdt[wm], y=pred_a[pm][wm], name='Model A (Absolute)',
    line=dict(color='#3498DB', width=1.5, dash='dash')))
fig.add_trace(go.Scatter(x=pdt[wm], y=pred_b[pm][wm], name='Model B (Residual)',
    line=dict(color='#9B59B6', width=1.5, dash='dash')))
fig.add_trace(go.Scatter(x=pdt[wm], y=pred_final[pm][wm], name='Ensemble ⚖️',
    line=dict(color='#FF6B6B', width=2.5)))
fig.add_hline(y=70, line_dash='dash', line_color='red')
fig.add_hline(y=180, line_dash='dash', line_color='orange')
fig.update_layout(title=f"📉 24h — A vs B vs Ensemble (Patient '{bp}')",
    template='plotly_dark', height=480, xaxis_title='Time', yaxis_title='mg/dL')
fig.show()

---\n## STEP 6 — Feature Importance (Both Models)

In [ ]:
def get_fi(mdl, feats):
    imp = mdl.get_booster().get_score(importance_type="gain")
    fi = {feats[int(k.replace('f',''))]: v for k,v in imp.items() if int(k.replace('f',''))<len(feats)}
    fi = dict(sorted(fi.items(), key=lambda x:x[1], reverse=True))
    tg = sum(fi.values())
    return {k: v/tg*100 for k,v in fi.items()}

fi_a = get_fi(model_a, FEATS)
fi_b = get_fi(model_b, FEATS)

print(f"{'Feature':30s} {'Model A':>8s} {'Model B':>8s} {'Delta':>8s}")
print("─" * 58)
for f in FEATS:
    a_p = fi_a.get(f, 0); b_p = fi_b.get(f, 0)
    flag = "⬇️" if f=="CGM" and b_p < a_p else ("⬆️" if b_p > a_p else "")
    print(f"  {f:28s} {a_p:>7.2f}% {b_p:>7.2f}% {b_p-a_p:>+7.2f}% {flag}")

print(f"\n🔍 CGM in Model A: {fi_a.get('CGM',0):.1f}%  |  Model B: {fi_b.get('CGM',0):.1f}%")

---\n## STEP 7 — Save Models & Report

In [ ]:
model_a.save_model(f"{MODEL_DIR}/ensemble_model_a_absolute.json")
model_b.save_model(f"{MODEL_DIR}/ensemble_model_b_residual.json")

config = {
    "ensemble": "hybrid_ensemble_v5",
    "created": datetime.now().isoformat(),
    "strategy": "weighted_average",
    "formula": f"w_a*pred_absolute + w_b*pred_residual",
    "model_a": {"file": "ensemble_model_a_absolute.json", "target": "Target_CGM"},
    "model_b": {"file": "ensemble_model_b_residual.json", "target": "Target_Residual",
                "reconstruction": "Linear_Forecast + pred_residual",
                "physics_baseline": "CGM + Vel_15min * 2"},
    "optimal_weight": {"w_a": best["w_a"], "w_b": best["w_b"]},
    "features": FEATS, "horizon_min": 30,
    "metrics": {
        "model_a_rmse": round(float(rmse_a),4),
        "model_b_rmse": round(float(rmse_b),4),
        "ensemble_rmse": round(float(rmse_f),4),
        "ensemble_mae": round(float(mae_f),4),
        "mape": round(float(mape_f),4),
        "clarke_ab_pct": round(float(ab_f),2),
        "zone_d_pct": round(float(zd_f),2),
    },
}
with open(f"{MODEL_DIR}/ensemble_config_v5.json", 'w') as f:
    json.dump(config, f, indent=2)

print(f"💾 Model A: ensemble_model_a_absolute.json")
print(f"💾 Model B: ensemble_model_b_residual.json")
print(f"📋 Config:  ensemble_config_v5.json")

In [ ]:
print("")
print("╔═══════════════════════════════════════════════════════════════════╗")
print("║            🏁  ENSEMBLE EVOLUTION REPORT                        ║")
print("╠═══════════════════════════════════════════════════════════════════╣")
print("║           v1(abs)  v3(delta) v4(phys)  v5(ensemble)             ║")
print("║  ─────────────────────────────────────────────────────────────   ║")
print(f"║  Clarke A+B: 92.91%    —        —       {ab_f:>6.2f}%               ║")
print(f"║  Zone D:      4.80%    —        —       {zd_f:>6.2f}%               ║")
print(f"║  RMSE:         —       —        —       {rmse_f:>8.4f}             ║")
print("║                                                                 ║")
print(f"║  Blend: {best['w_a']:.0%} Absolute + {best['w_b']:.0%} Residual                       ║")
print(f"║  Model A alone: RMSE={rmse_a:.4f}                               ║")
print(f"║  Model B alone: RMSE={rmse_b:.4f}                               ║")
print("║                                                                 ║")
v1 = "✅ TARGET MET" if ab_f > 95 else "⚠️  Below target"
v2 = "✅ SAFE" if zd_f < 3 else "⚠️  Zone D high"
print(f"║  CLARKE:  {v1:53s} ║")
print(f"║  SAFETY:  {v2:53s} ║")
print("╚═══════════════════════════════════════════════════════════════════╝")
gc.collect()
print("\n🧹 Done.")